# Phase 2 — LegalIR Harrier on RTX Pro 6000 (offline)
Attach the Phase 2 Harrier bundle and a competition-data dataset containing `train.json`, `private-official.json`, and `selected-contexts/selected-contexts/`. Select **RTX Pro 6000** and set Internet to **Off**. By default this runs inference on the private questions; it does not calculate private Recall because the labels are unavailable. The test-specific working directory is preserved when rerunning cells in the same live session so cached pipeline work can resume.

In [ ]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from pathlib import Path
EXPERIMENT_ID = 'phase2-vietlegal-harrier-0.6b'
DATASET_DIR = Path('/kaggle/input/REPLACE_WITH_COMPETITION_DATASET_SLUG')
TEST_FILENAME = 'private-official.json'  # switch to public-official.json for public inference
if TEST_FILENAME not in {'private-official.json', 'public-official.json'}:
    raise ValueError(f'Unsupported test input: {TEST_FILENAME}')
TEST_LABEL = 'private' if TEST_FILENAME == 'private-official.json' else 'public'
CHECKPOINT_ARTIFACTS_DIR = None  # e.g. Path('/kaggle/input/phase2-public-checkpoint/artifacts_phase2_harrier')
BUNDLE_DIR = Path('/kaggle/input/REPLACE_WITH_PHASE2_HARRIER_BUNDLE_SLUG/legalir-phase2-harrier-bundle')
WORK_DIR = Path(f'/kaggle/working/legalir-phase2-{TEST_LABEL}-harrier-run')
RUNTIME_DIR = Path('/kaggle/working/legalir-phase2-harrier-runtime')

In [ ]:
import json
import shutil
import subprocess
import sys
import time

def run(*command, cwd=None, env=None):
    print('+', ' '.join(map(str, command)))
    started = time.perf_counter()
    subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)
    print(f'Completed in {(time.perf_counter() - started) / 60:.1f} minutes')

manifest_path = BUNDLE_DIR / 'manifests' / 'bundle_manifest.json'
if not manifest_path.is_file():
    raise FileNotFoundError(f'Missing Phase 2 manifest: {manifest_path}')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
if manifest.get('experiment_id') != EXPERIMENT_ID:
    raise RuntimeError(f"Wrong bundle: expected {EXPERIMENT_ID}, found {manifest.get('experiment_id')}")
model_names = [model['name'] for model in manifest['models']]
if 'vietlegal_harrier' not in model_names or 'vietlegal_e5' in model_names:
    raise RuntimeError(f'Bundle has incorrect Phase 2 model set: {model_names}')
for record in manifest['files']:
    path = BUNDLE_DIR / record['path']
    if not path.is_file() or path.stat().st_size != record['bytes']:
        raise RuntimeError(f'Bundle file missing or truncated: {path}')
required_wheel_prefixes = ('sentence_transformers-5.7.0-', 'transformers-5.17.0-', 'tokenizers-0.23.2-', 'safetensors-0.8.0-')
wheel_names = [path.name.lower() for path in (BUNDLE_DIR / 'wheels').glob('*.whl')]
missing_runtime_wheels = [prefix for prefix in required_wheel_prefixes if not any(name.startswith(prefix) for name in wheel_names)]
if missing_runtime_wheels:
    raise RuntimeError(f'Phase 2 bundle is stale; missing wheels: {missing_runtime_wheels}')
if RUNTIME_DIR.exists():
    shutil.rmtree(RUNTIME_DIR)
RUNTIME_DIR.mkdir(parents=True)
run(sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--ignore-installed', '--target', RUNTIME_DIR, '--find-links', BUNDLE_DIR / 'wheels', '-r', BUNDLE_DIR / 'requirements-offline.txt')
project_wheels = sorted((BUNDLE_DIR / 'wheels').glob('uit_legalir-*.whl'))
if len(project_wheels) != 1:
    raise RuntimeError(f'Expected exactly one project wheel, found {project_wheels}')
run(sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--ignore-installed', '--target', RUNTIME_DIR, project_wheels[0])
runtime_env = os.environ.copy()
runtime_env['PYTHONPATH'] = str(RUNTIME_DIR)
runtime_env['PYTHONNOUSERSITE'] = '1'
version_probe = """
from importlib.metadata import version
expected = {'sentence-transformers': '5.7.0', 'transformers': '5.17.0', 'tokenizers': '0.23.2', 'safetensors': '0.8.0'}
actual = {name: version(name) for name in expected}
if actual != expected:
    raise RuntimeError(f'Incorrect isolated runtime versions: expected {expected}, found {actual}')
print('Pinned Phase 2 runtime:', actual)
"""
run(sys.executable, '-c', version_probe, env=runtime_env)
print('Phase 2 bundle project commit:', manifest['project_commit'])

In [ ]:
gpu_probe = ("import torch; assert torch.cuda.is_available(), 'No CUDA GPU is available. Select RTX Pro 6000.'; "
             "print('GPU:', torch.cuda.get_device_name(0)); print('CUDA:', torch.version.cuda, 'capability:', torch.cuda.get_device_capability(0))")
run(sys.executable, '-c', gpu_probe, env=runtime_env)
contexts_source = DATASET_DIR / 'selected-contexts' / 'selected-contexts'
if not contexts_source.is_dir():
    raise FileNotFoundError(f'Missing nested competition corpus directory: {contexts_source}')
context_count = sum(1 for _ in contexts_source.glob('context_*.json'))
if not context_count:
    raise FileNotFoundError(f'No context files found in {contexts_source}')
for filename in ('train.json', TEST_FILENAME):
    if not (DATASET_DIR / filename).is_file():
        raise FileNotFoundError(f'Missing competition input: {DATASET_DIR / filename}')
WORK_DIR.mkdir(parents=True, exist_ok=True)
def ensure_input_link(destination, source, is_directory=False):
    if destination.is_symlink():
        if destination.resolve() == source.resolve(): return
        destination.unlink()
    elif destination.exists():
        raise RuntimeError(f'Refusing to overwrite existing work file: {destination}')
    destination.symlink_to(source, target_is_directory=is_directory)
ensure_input_link(WORK_DIR / 'selected-contexts', contexts_source, is_directory=True)
for filename in ('train.json', TEST_FILENAME): ensure_input_link(WORK_DIR / filename, DATASET_DIR / filename)
test_bytes = (WORK_DIR / TEST_FILENAME).read_bytes()
test_question_count = len(json.loads(test_bytes))
test_fingerprint = __import__('hashlib').sha256(test_bytes).hexdigest()
print(f'Using {context_count} contexts and {test_question_count} {TEST_LABEL} questions from {TEST_FILENAME} in {WORK_DIR}')

In [ ]:
import yaml
config = yaml.safe_load((BUNDLE_DIR / 'configs' / 'kaggle_rtx_pro_6000.yaml').read_text(encoding='utf-8'))
if 'vietlegal_harrier' not in config['models'] or 'vietlegal_e5' in config['models']:
    raise RuntimeError('Attached config is not the isolated Phase 2 Harrier configuration')
config['paths']['public_file'] = TEST_FILENAME  # the pipeline calls this split 'public' internally
for name in config['models']:
    config['models'][name]['local_path'] = str(BUNDLE_DIR / 'models' / name)
    config['models'][name]['local_files_only'] = True
artifacts = WORK_DIR / config['paths']['artifacts_dir']
artifacts.mkdir(parents=True, exist_ok=True)
def seed_checkpoint(source_dir):
    if source_dir is None: return
    source_dir = Path(source_dir).resolve()
    if not source_dir.is_dir(): raise FileNotFoundError(f'Missing checkpoint artifacts: {source_dir}')
    prepare_checkpoint = source_dir / 'prepare_manifest.json'
    if not prepare_checkpoint.is_file(): raise RuntimeError(f'Checkpoint has no prepare manifest: {prepare_checkpoint}')
    prepare_metadata = json.loads(prepare_checkpoint.read_text(encoding='utf-8'))
    expected_chunking = __import__('hashlib').sha256(json.dumps(config['chunking'], ensure_ascii=False, sort_keys=True).encode('utf-8')).hexdigest()
    if prepare_metadata.get('chunking_fingerprint') != expected_chunking: raise RuntimeError('Checkpoint chunking config does not match this run')
    checkpoint_manifest = source_dir / 'model_manifest.json'
    checkpoint_names = set()
    if checkpoint_manifest.is_file(): checkpoint_names = {row['name'] for row in json.loads(checkpoint_manifest.read_text(encoding='utf-8'))['models']}
    dense_names = {'vietlegal_harrier', 'vietnamese_embedding', 'nemotron'}
    if checkpoint_names and not dense_names <= checkpoint_names: raise RuntimeError(f'Checkpoint lacks required dense retrievers: {checkpoint_names}')
    for name in dense_names:
        for required in ('vectors.npy', 'index.faiss', 'chunks.json'):
            if not (source_dir / 'dense' / name / required).is_file(): raise RuntimeError(f'Incomplete dense checkpoint: {name}/{required}')
        for required in ('vectors.npy', 'questions.json'):
            if not (source_dir / 'question_memory' / name / required).is_file(): raise RuntimeError(f'Incomplete question-memory checkpoint: {name}/{required}')
    for relative in ('corpus.jsonl', 'chunks_short.jsonl', 'chunks_long.jsonl', 'lexical_short.pkl', 'dense', 'question_memory'):
        source = source_dir / relative
        destination = artifacts / relative
        if source.exists() and not destination.exists(): destination.symlink_to(source, target_is_directory=source.is_dir())
    same_phase = checkpoint_names == set(config['models'])
    patterns = ['prepare_manifest.json', 'train_questions.jsonl']
    if same_phase: patterns += ['first_stage_weights.json', 'final_weights*.json', 'retrieval_train.json', 'fused_train.json', 'rerank_train*.json']
    for pattern in patterns:
        for source in source_dir.glob(pattern):
            destination = artifacts / source.name
            if not destination.exists(): shutil.copy2(source, destination)
    print('Seeded', 'Phase 2' if same_phase else 'retrieval-only', 'checkpoint from:', source_dir)
seed_checkpoint(CHECKPOINT_ARTIFACTS_DIR)
input_state_path = WORK_DIR / 'inference_input_state.json'
config_fingerprint = __import__('hashlib').sha256(yaml.safe_dump(config, allow_unicode=True, sort_keys=True).encode('utf-8')).hexdigest()
input_state = {'filename': TEST_FILENAME, 'sha256': test_fingerprint, 'questions': test_question_count, 'config_sha256': config_fingerprint, 'project_commit': manifest['project_commit']}
previous_input_state = json.loads(input_state_path.read_text(encoding='utf-8')) if input_state_path.is_file() else None
if previous_input_state != input_state:
    for pattern in ('public_questions.jsonl', 'retrieval_public.json', 'fused_public.json', 'rerank_public*.json'):
        for stale in artifacts.glob(pattern):
            if stale.is_file() or stale.is_symlink(): stale.unlink()
    print('Invalidated test-dependent caches:', previous_input_state, '->', input_state)
input_state_path.write_text(json.dumps(input_state, ensure_ascii=False, indent=2), encoding='utf-8')
config_path = WORK_DIR / 'kaggle_rtx_pro_6000_phase2_harrier.yaml'
config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')
print(config_path.read_text(encoding='utf-8'))

In [ ]:
preflight_path = WORK_DIR / 'offline_preflight_phase2.py'
preflight_path.write_text("""
import sys
from pathlib import Path
import torch
import yaml
from legalir.embeddings import load_encoder
from legalir.rerank import JinaListwiseReranker, VietnamesePairwiseReranker
config = yaml.safe_load(Path(sys.argv[1]).read_text(encoding='utf-8'))
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('high')
for name, spec in config['models'].items():
    if spec['role'] != 'dense':
        continue
    model = load_encoder(spec, config['runtime'])
    probe = spec['prompt_query'] + 'điều kiện cấp giấy phép'
    assert len(model.encode([probe], convert_to_numpy=True)) == 1
    del model
    torch.cuda.empty_cache()
pairwise = VietnamesePairwiseReranker(config)
assert len(pairwise.rank('câu hỏi', ['văn bản một', 'văn bản hai'])) == 2
pairwise.close()
jina = JinaListwiseReranker(config)
assert len(jina.rank('câu hỏi', ['văn bản một', 'văn bản hai'])) == 2
jina.close()
print('All Phase 2 local model preflight tests passed.')
""".lstrip(), encoding='utf-8')
run(sys.executable, preflight_path, config_path, cwd=WORK_DIR, env=runtime_env)

In [ ]:
base = [sys.executable, '-m', 'legalir']
def legalir(*args):
    run(*base, *args, cwd=WORK_DIR, env=runtime_env)

dense_models = [name for name, spec in config['models'].items() if spec['role'] == 'dense']
print('Phase 2 dense retrievers:', dense_models)
legalir('prepare', '--config', config_path, '--resume')
legalir('audit', '--config', config_path)
legalir('index', '--config', config_path, '--lexical-only', '--resume')
for model in dense_models:
    legalir('index', '--config', config_path, '--model', model, '--resume')
legalir('tune', '--config', config_path, '--resume')
for fold in config['validation']['reranker_tuning_folds']:
    for engine in ('vietnamese_reranker', 'jina'):
        legalir('rerank', '--config', config_path, '--split', 'train', '--fold', str(fold), '--engine', engine, '--resume')
    legalir('rerank', '--config', config_path, '--split', 'train', '--fold', str(fold), '--resume')
legalir('tune', '--config', config_path, '--final', '--resume')
legalir('retrieve', '--config', config_path, '--split', 'public', '--resume')
for engine in ('vietnamese_reranker', 'jina'):
    legalir('rerank', '--config', config_path, '--split', 'public', '--engine', engine, '--resume')
submission_json = WORK_DIR / f'submission_phase2_{TEST_LABEL}_harrier.json'
submission_zip = WORK_DIR / f'submission_phase2_{TEST_LABEL}_harrier.zip'
legalir('predict', '--config', config_path, '--output', submission_json, '--resume')
run('zip', '-j', submission_zip, submission_json)
print('Phase 2 submission:', submission_zip)

In [ ]:
first_stage = json.loads((artifacts / 'first_stage_weights.json').read_text(encoding='utf-8'))
final_stage = json.loads((artifacts / 'final_weights.json').read_text(encoding='utf-8'))
model_audit = json.loads((artifacts / 'model_manifest.json').read_text(encoding='utf-8'))
report = {
    'experiment_id': EXPERIMENT_ID,
    'test_file': TEST_FILENAME,
    'test_questions': test_question_count,
    'test_sha256': test_fingerprint,
    'evaluation': 'inference_only; labels and Recall not evaluated',
    'project_commit': manifest['project_commit'],
    'dense_models': [config['models'][name]['id'] for name in dense_models],
    'total_parameters': model_audit['total_parameters'],
    'candidate_metrics': first_stage.get('candidate_metrics'),
    'oof_final_metrics': final_stage.get('metrics'),
    'oof_fold_metrics': final_stage.get('oof_fold_metrics'),
    'submission': str(submission_zip),
}
report_path = WORK_DIR / 'phase2_harrier_report.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(report, ensure_ascii=False, indent=2))